# Production / distribution: a minimum-cost flow problem

Python (Pyomo) version of the example in the slides `1 Lecture/Production Distribution Model.pdf`
(translated from the MATLAB script `5 Source Files/MATLAB/production_distribution_model.m`). It is also a worked **transhipment / network flow** example
for Semester 1 Week 8.

A car maker has two plants. Plant 1 makes models 1 and 2; plant 2 makes models 1, 2 and 3. Cars are
shipped to two retailers. Choose how many of each model to make at each plant and where to ship them
so that every retailer's demand is met at minimum production + transport cost.

In [ ]:
# Run this cell first (installs the extra packages; safe to re-run).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

**Data.** Plant capacities and production costs per car, retailer demands, and transport costs per car for each route that exists. Plant 1 cannot make model 3.

In [2]:
import pyomo.environ as pyo

plants, models, retailers = ["P1", "P2"], ["M1", "M2", "M3"], ["R1", "R2"]

# production capacity (cars) and cost (£ per car); plant 1 cannot make model 3
capacity = {("P1", "M1"): 150, ("P1", "M2"): 120,
            ("P2", "M1"): 100, ("P2", "M2"): 150, ("P2", "M3"): 100}
prod_cost = {("P1", "M1"): 1000, ("P1", "M2"): 1200,
             ("P2", "M1"): 900, ("P2", "M2"): 1100, ("P2", "M3"): 1500}

# demand (cars) at each retailer
demand = {("R1", "M1"): 100, ("R1", "M2"): 80, ("R1", "M3"): 70,
          ("R2", "M1"): 100, ("R2", "M2"): 130}

# transport cost (£ per car) for each (plant, model, retailer) route that exists
ship_cost = {("P1", "M1", "R1"): 4, ("P1", "M1", "R2"): 5,
             ("P1", "M2", "R1"): 3, ("P1", "M2", "R2"): 6,
             ("P2", "M1", "R1"): 5, ("P2", "M1", "R2"): 6,
             ("P2", "M2", "R1"): 4, ("P2", "M2", "R2"): 3,
             ("P2", "M3", "R1"): 4}

**Model and solve.** Variables: cars made at each plant (per model) and cars shipped on each route. Minimise production + transport cost, subject to capacity, flow balance at each plant, and demand.

In [3]:
m = pyo.ConcreteModel()
m.PM = pyo.Set(initialize=list(capacity), dimen=2)          # plant-model pairs that exist
m.ROUTES = pyo.Set(initialize=list(ship_cost), dimen=3)     # (plant, model, retailer)
m.RM = pyo.Set(initialize=list(demand), dimen=2)            # retailer-model demands

m.make = pyo.Var(m.PM, within=pyo.NonNegativeReals)         # cars produced
m.ship = pyo.Var(m.ROUTES, within=pyo.NonNegativeReals)     # cars shipped

m.cost = pyo.Objective(
    expr=sum(prod_cost[k] * m.make[k] for k in m.PM) + sum(ship_cost[r] * m.ship[r] for r in m.ROUTES),
    sense=pyo.minimize)

# capacity at each plant for each model
m.cap = pyo.Constraint(m.PM, rule=lambda m, p, mod: m.make[p, mod] <= capacity[p, mod])

# flow balance at each plant: everything made is shipped
m.balance = pyo.Constraint(m.PM, rule=lambda m, p, mod:
    m.make[p, mod] == sum(m.ship[p, mod, r] for r in retailers if (p, mod, r) in ship_cost))

# demand at each retailer
m.meet = pyo.Constraint(m.RM, rule=lambda m, r, mod:
    sum(m.ship[p, mod, r] for p in plants if (p, mod, r) in ship_cost) >= demand[r, mod])

result = pyo.SolverFactory("appsi_highs").solve(m)
print("Status:", result.solver.termination_condition)
print(f"Minimum total cost: £{pyo.value(m.cost):,.0f}")

Status: optimal
Minimum total cost: £533,930


**The plan.** Expected: total £533,930. Plant 1 makes 100 of model 1 and 60 of model 2; plant 2 makes 100, 150 and 70. This matches the annotated slide 11 and an independent check with SciPy.

In [4]:
print("Production plan")
for (p, mod) in m.PM:
    print(f"  {p} {mod}: {m.make[p, mod].value:6.0f} cars  (capacity {capacity[p, mod]})")

print("\nShipping plan")
for (p, mod, r) in m.ROUTES:
    if m.ship[p, mod, r].value > 1e-6:
        print(f"  {p} -> {r}  {mod}: {m.ship[p, mod, r].value:6.0f} cars")

Production plan
  P1 M1:    100 cars  (capacity 150)
  P1 M2:     60 cars  (capacity 120)
  P2 M1:    100 cars  (capacity 100)
  P2 M2:    150 cars  (capacity 150)
  P2 M3:     70 cars  (capacity 100)

Shipping plan
  P1 -> R2  M1:    100 cars
  P1 -> R1  M2:     60 cars
  P2 -> R1  M1:    100 cars
  P2 -> R1  M2:     20 cars
  P2 -> R2  M2:    130 cars
  P2 -> R1  M3:     70 cars
